In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import joblib

In [2]:
import pandas as pd
use_cols = ['Created Date','Closed Date','Complaint Type','Agency','Borough','Descriptor']
df = pd.read_csv('../dsml-project-team/311-service-requests-from-2010-to-present.csv', usecols=use_cols, nrows=500000)


In [3]:
df['Created Date'] = pd.to_datetime(df['Created Date'], errors='coerce')
df['Closed Date']  = pd.to_datetime(df['Closed Date'], errors='coerce')


In [4]:
# Handle missing values
df['Complaint Type'] = df['Complaint Type'].fillna('')
df['Descriptor'] = df['Descriptor'].fillna('')
df['Borough'] = df['Borough'].fillna('Unknown')

In [5]:
# Combine complaint type and descriptor
df['text'] = df['Complaint Type'] + " " + df['Descriptor']

In [14]:
import re, nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
nltk.download('stopwords'); nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    if pd.isna(text): return ""
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = [lemmatizer.lemmatize(word) for word in text.split() if word not in stop_words]
    return " ".join(tokens)

df['clean_text'] = df['Complaint Type'].astype(str).apply(clean_text)


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [15]:
# Text Cleaning

def clean_text(text):
    text = str(text).lower()  # Convert to lowercase
    
    text = re.sub(r"http\S+|www\S+", '', text)  # Remove URLs

    
    text = re.sub(r"[^a-z\s]", '', text)  # Remove special characters
    
    words = text.split()
    
    # Remove stopwords + apply lemmatization
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    
    return " ".join(words)

In [16]:

df['clean_text'] = df['text'].apply(clean_text)

df[['text', 'clean_text']].head()

,text,clean_text
0,Street Condition Pothole,street condition pothole
1,Noise - Commercial Loud Music/Party,noise commercial loud musicparty
2,Noise - Residential Loud Music/Party,noise residential loud musicparty
3,Noise - Residential Loud Music/Party,noise residential loud musicparty
4,Illegal Parking Commercial Overnight Parking,illegal parking commercial overnight parking


In [17]:
from textblob import TextBlob

def get_sentiment(text):
    if not text or pd.isna(text):
        return "Neutral"
    polarity = TextBlob(text).sentiment.polarity
    if polarity > 0.3:
        return "Positive"
    elif polarity < -0.3:
        return "Negative"
    else:
        return "Neutral"

df['sentiment'] = df['clean_text'].apply(get_sentiment)


In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import joblib

In [19]:
# TF-IDF
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X = vectorizer.fit_transform(df['clean_text'])
y = df['Agency']

In [24]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

# Features: TF-IDF from Week 2
X_text = df['clean_text'].astype(str)
y = df['sentiment']   # Positive / Negative / Neutral

tfidf_sent = TfidfVectorizer(stop_words='english', max_features=5000)
X = tfidf_sent.fit_transform(X_text)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

sentiment_model = MultinomialNB()
sentiment_model.fit(X_train, y_train)

y_pred = sentiment_model.predict(X_test)
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

    Negative       0.96      1.00      0.98     13331
     Neutral       1.00      0.99      1.00     84137
    Positive       0.95      1.00      0.97      2532

    accuracy                           0.99    100000
   macro avg       0.97      1.00      0.98    100000
weighted avg       0.99      0.99      0.99    100000



In [25]:
urgency_map = {
    "Positive": 1,
    "Neutral": 2,
    "Negative": 3,
    "Critical": 4   # if you add a Critical class later
}

df['urgency_score'] = df['sentiment'].map(urgency_map)


In [28]:
import joblib

joblib.dump(sentiment_model, "sentiment_classifier.pkl")
joblib.dump(tfidf_sent, "tfidf_vectorizer_sent.pkl")

['tfidf_vectorizer_sent.pkl']